# Sorami Phase 2 Training (Colab)

**사용법:**
1. Google Drive에 타겟 여성 목소리 녹음을 `/MyDrive/sorami_data/target_female/` 에 업로드
2. 이 노트북을 열고 Runtime → Change runtime type → GPU (T4 또는 A100) 선택
3. 모든 셀을 순서대로 실행 (Ctrl+F9)
4. 학습 완료 후 `runs/phase2/best.pt`를 다운로드

## 1. 환경 설정 & 의존성 설치

In [ ]:
!pip install -q torch torchaudio pyyaml pyworld pyloudnorm librosa
!pip install -q onnx onnxruntime

In [ ]:
# 리포지토리 clone
!git clone https://github.com/nyumigi-cpu/Rvc_Sorami.git
%cd Rvc_Sorami

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

## 2. 데이터 경로 설정

In [ ]:
import os
from pathlib import Path

# 드라이브의 데이터 경로 (본인에 맞게 수정)
DRIVE_DATA = Path('/content/drive/MyDrive/sorami_data/target_female')
assert DRIVE_DATA.exists(), f'데이터 경로가 없습니다: {DRIVE_DATA}'

# 로컬 심볼릭 링크
os.makedirs('./data', exist_ok=True)
local_data = Path('./data/target_female')
if not local_data.exists():
    os.symlink(DRIVE_DATA, local_data)

# 오디오 파일 수 확인
wavs = list(local_data.rglob('*.wav')) + list(local_data.rglob('*.flac'))
print(f'오디오 파일 수: {len(wavs)}')

## 3. Phase 2 학습 시작

In [ ]:
!python -m training.train --phase 2 --config training/configs/phase2_gan.yaml

## 4. 체크포인트 드라이브로 복사

In [ ]:
import shutil
drive_out = Path('/content/drive/MyDrive/sorami_checkpoints/phase2')
drive_out.mkdir(parents=True, exist_ok=True)
for ckpt in Path('./runs/phase2').glob('*.pt'):
    shutil.copy(ckpt, drive_out / ckpt.name)
    print(f'복사: {ckpt.name}')

## 5. ONNX 내보내기 + INT8 양자화

In [ ]:
!python -m training.export.export_onnx --ckpt ./runs/phase2/best.pt --output ./onnx_out --segment 1.0
!python -m training.export.quantize --input ./onnx_out

In [ ]:
# ONNX 모델도 드라이브로
drive_onnx = Path('/content/drive/MyDrive/sorami_checkpoints/onnx')
drive_onnx.mkdir(parents=True, exist_ok=True)
for f in Path('./onnx_out').glob('*'):
    shutil.copy(f, drive_onnx / f.name)
    print(f'복사: {f.name}')